# Metadata-Driven ETL Framework Overview

I'll analyze the metadata tables structure and create an ER diagram showing their relationships and data flow.

---

# 📊 ER Diagram — Table Relationships

Your metadata framework has **two distinct domains**:

## 1. Configuration Domain (Design Time)

Tables that define **what** and **how** to process.

### `table_config`
Central registry table.

**Purpose**
- Source paths
- Target locations
- Drift policies
- Load priorities

**Primary Key**
- `table_id`

---

### `table_process_config`
Process definitions table.

**Relationship**
- Links to `table_config` via `table_id` (FK)

**Purpose**
- Defines Bronze/Silver/Gold processes
- Process dependencies
- Load types

---

### `input_column_config`
Column-level metadata table.

**Relationship**
- Links to `table_config` via `table_id` (FK)

**Purpose**
- Schema enforcement
- PII flags
- Transformations
- Column purpose definitions

---

## 2. Runtime Domain (Execution Time)

Tables that track **execution** and **observability**.

### `etl_process_master`
Pipeline run tracker.

**Primary Key**
- `run_id`

**Purpose**
- Tracks pipeline-level metrics
- Run status
- Row counts
- Execution duration

---

### `etl_process_log`
Table-level execution logs.

**Relationships**
- Links to `etl_process_master` via `run_id` (FK)
- Links to `table_config` via `table_id` (FK)

**Purpose**
- Tracks rows read/written
- Delta versions
- Execution duration
- Error details

---

### `schema_drift_log`
Schema evolution audit table.

**Relationships**
- Links to `etl_process_master` via `run_id` (FK)
- Links to `table_config` via `table_id` (FK)

**Purpose**
- Logs schema drift events:
  - NEW_COLUMN
  - MISSING_COLUMN
  - TYPE_CHANGED
- Stores corrective actions taken

---

# 🔄 Data Flow — End-to-End Execution

## Phase 1: Configuration Setup

1. Populate `table_config` with source-to-target mappings
2. Define processes in `table_process_config`
   - Bronze → Silver → Gold
3. Configure column schemas in `input_column_config`

---

## Phase 2: Pipeline Execution

### Step 1 — Pipeline Trigger
Pipeline starts (scheduled/manual) and creates a `run_id` in `etl_process_master`.

### Step 2 — Table Processing
For each table (ordered using `load_priority` and `depends_on_table_ids`):

- Log process start in `etl_process_log`
- Read metadata from:
  - `table_config`
  - `table_process_config`
  - `input_column_config`
- Execute ingestion/transformation
- Apply schema enforcement
- Detect schema drift
- Write drift events to `schema_drift_log`
- Update execution metrics in `etl_process_log`

### Step 3 — Pipeline Completion
Update `etl_process_master` with:
- Final run status
- End timestamp
- Aggregate metrics

---

## Phase 3: Observability & Monitoring

Runtime tables enable monitoring of:

- Pipeline success/failure trends
- Table-level bottlenecks
- Processing duration analysis
- Data volume growth
- Schema evolution history
- Data quality and drift patterns

---

# 🔗 Key Relationships

## Runtime Logs → Configuration

- `etl_process_log.table_id`
  → `table_config.table_id`

- `schema_drift_log.table_id`
  → `table_config.table_id`

---

## Runtime Logs → Pipeline Run Context

- `etl_process_log.run_id`
  → `etl_process_master.run_id`

- `schema_drift_log.run_id`
  → `etl_process_master.run_id`

---

## Configuration Hierarchy

### One-to-Many Relationships

- `table_config (1)`
  → `table_process_config (N)`

- `table_config (1)`
  → `input_column_config (N)`

---

# 🧠 Architectural Insight

This is a **metadata-driven declarative ETL architecture**.

Key advantage:

> Adding a new table typically requires only metadata configuration changes, not pipeline code modifications.

That gives:
- Better scalability
- Reduced engineering effort
- Centralized governance
- Faster onboarding of new datasets
- Easier schema evolution management

---

# 🏗 Suggested High-Level ER Relationship View

```text
                    +----------------------+
                    |     table_config     |
                    |----------------------|
                    | table_id (PK)        |
                    +----------+-----------+
                               |
               +---------------+----------------+
               |                                |
               |                                |
               v                                v

+---------------------------+     +---------------------------+
|   table_process_config    |     |    input_column_config   |
|---------------------------|     |---------------------------|
| process_id (PK)           |     | column_id (PK)            |
| table_id (FK)             |     | table_id (FK)             |
+---------------------------+     +---------------------------+


                    +----------------------+
                    |  etl_process_master  |
                    |----------------------|
                    | run_id (PK)          |
                    +----------+-----------+
                               |
               +---------------+----------------+
               |                                |
               |                                |
               v                                v

+---------------------------+     +---------------------------+
|     etl_process_log       |     |     schema_drift_log     |
|---------------------------|     |---------------------------|
| log_id (PK)               |     | drift_id (PK)             |
| run_id (FK)               |     | run_id (FK)               |
| table_id (FK)             |     | table_id (FK)             |
+---------------------------+     +---------------------------+
```

--- 

In [0]:
00_architecture_overview
│
├── Medallion Architecture
├── Metadata Framework
├── ER Diagram
├── Runtime Logging Flow
├── Schema Drift Handling
├── Orchestration Strategy
└── Embedded PNG diagrams